# Профилилирование и оптимизация выполнения кода

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы:
* Макрушин С.В. "Оптимизация выполнения кода, векторизация, Numba"
* IPython Cookbook, Second Edition (2018), глава 4
* https://ipython-books.github.io/43-profiling-your-code-line-by-line-with-line_profiler/

## Задачи для совместного разбора

In [ ]:
!pip install line_profiler
# !pip install --user numpy==1.20

In [ ]:
import numpy as np

1. Сгенерируйте массив `A` из `N=1млн` случайных целых чисел на отрезке от 0 до 1000. Пусть `B[i] = A[i] + 100`. Посчитайте среднее значение массива `B`.

In [ ]:
A = np.random.randint(0, 1001, 1000000)

In [ ]:
def nice_function(A):
    acc, cnt = 0, 0
    for a in A:
        b = a + 100
        acc += b
        cnt += 1
    return acc / cnt

In [ ]:
nice_function(A)

599.718788

In [ ]:
%%time
# на уровне ячейки
nice_function(A)

CPU times: user 234 ms, sys: 1.52 ms, total: 235 ms
Wall time: 235 ms


599.718788

In [ ]:
%time nice_function(A)
# На уровне строки

CPU times: user 236 ms, sys: 1.72 ms, total: 238 ms
Wall time: 238 ms


599.718788

In [ ]:
%%timeit # более точное замерение через несколько ранов и вычленения среднего
nice_function(A)

234 ms ± 1.12 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
%load_ext line_profiler
# для того чтобы пользоваться магической %lprun

The line_profiler extension is already loaded. To reload it, use:
  %reload_ext line_profiler


In [ ]:
%lprun -f nice_function nice_function(A) #исследование время выполнения каждой строчки функции nice_function

Timer unit: 1e-09 s

Total time: 0.851033 s
File: <ipython-input-133-97f65c44397e>
Function: nice_function at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def nice_function(A):
     2         1       2000.0   2000.0      0.0      acc, cnt = 0, 0
     3   1000000  202935000.0    202.9     23.8      for a in A:
     4   1000000  278409000.0    278.4     32.7          b = a + 100
     5   1000000  204392000.0    204.4     24.0          acc += b
     6   1000000  165293000.0    165.3     19.4          cnt += 1
     7         1       2000.0   2000.0      0.0      return acc / cnt

In [ ]:
def nice_function2(A):
    acc, cnt = 0, len(A)
    for a in A:
        acc += a
    return acc / cnt + 100

In [ ]:
%lprun -f nice_function2 nice_function2(A)

Timer unit: 1e-09 s

Total time: 0.409444 s
File: <ipython-input-140-1d617987449e>
Function: nice_function2 at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def nice_function2(A):
     2         1       1000.0   1000.0      0.0      acc, cnt = 0, len(A)
     3   1000000  201667000.0    201.7     49.3      for a in A:
     4   1000000  207759000.0    207.8     50.7          acc += a
     5         1      17000.0  17000.0      0.0      return acc / cnt + 100

In [ ]:
def nice_function3(A):
    acc, cnt = 0, len(A)
    for a in A:
        acc += a
    return acc / cnt + 100

In [ ]:
%%timeit
nice_function3(A)

86.5 ms ± 541 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
%%timeit
A.mean() + 100

530 µs ± 2.5 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [ ]:
A = np.random.randint(0, 1001, 1000000)
B = A + 100
B.mean()

600.185769

2. Создайте таблицу 2млн строк и с 4 столбцами, заполненными случайными числами. Добавьте столбец `key`, которые содержит элементы из множества английских букв. Выберите из таблицы подмножество строк, для которых в столбце `key` указаны первые 5 английских букв.

In [ ]:
import pandas as pd
import string

N = 2_000_000
df = pd.DataFrame(np.random.randn(N, 4), columns=[f"col{i}" for i in range(4)])
df["key"] = np.random.choice(list(string.ascii_letters.lower()), N, replace=True)
df.query('key in ("a", "b", "c", "d", "e")').head()
#df.loc[df['key'].isin(('a', 'b', 'c', 'd', 'e'))]

,col0,col1,col2,col3,key
0,1.066152,-1.222764,-0.709771,0.450522,e
4,0.637939,0.190611,-0.479804,0.902168,b
12,1.653588,0.342662,-1.035477,0.857549,a
15,-1.025890,1.635620,0.588261,1.307955,a
19,0.686779,-0.646883,-2.217900,1.786937,b


In [ ]:
%%timeit
df.query('key in ("a", "b", "c", "d", "e")')

85.9 ms ± 1.25 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
%%timeit
df.loc[df['key'].isin(('a', 'b', 'c', 'd', 'e'))]

81.1 ms ± 305 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
def g(df): # это сразу гг
    mask = []
    for _, row in df.iterrows():
        if row['key'] in {'a', 'b', 'c', 'd', 'e'}:
            mask.append(True)
        else:
            mask.append(False)
    r = df[mask]
    return r

In [ ]:
%%time
g(df)

CPU times: user 1min 8s, sys: 530 ms, total: 1min 9s
Wall time: 1min 10s


,col0,col1,col2,col3,key
0,1.066152,-1.222764,-0.709771,0.450522,e
4,0.637939,0.190611,-0.479804,0.902168,b
12,1.653588,0.342662,-1.035477,0.857549,a
15,-1.025890,1.635620,0.588261,1.307955,a
19,0.686779,-0.646883,-2.217900,1.786937,b
...,...,...,...,...,...
1999976,-0.877272,-1.978170,-0.895496,-0.313471,d
1999977,0.778031,0.924455,-0.375220,-0.413089,d
1999985,0.393546,1.215272,-0.410780,0.259987,d
1999991,-0.816180,0.228803,2.079133,-0.943553,d


## Лабораторная работа 1

__При решении данных задач не подразумевается использования циклов или генераторов Python в ходе работы с пакетами `numpy` и `pandas`, если в задании не сказано обратного. Решения задач, в которых для обработки массивов `numpy` или структур `pandas` используются явные циклы (без согласования с преподавателем), могут быть признаны некорректными и не засчитаны.__

В файлах `recipes_sample.csv` и `reviews_sample.csv` находится информация об рецептах блюд и отзывах на эти рецепты соответственно. Загрузите данные из файлов в виде `pd.DataFrame` с названиями `recipes` и `reviews`. Обратите внимание на корректное считывание столбца(ов) с индексами. Приведите столбцы к нужным типам.

In [ ]:
recipes = pd.read_csv('/Users/ivanlopatkin/Downloads/recipes_sample (1).csv')
reviews = pd.read_csv('/Users/ivanlopatkin/Downloads/reviews_sample.csv', index_col=0)
recipes.head()

,name,id,minutes,contributor_id,submitted,n_steps,description,n_ingredients
0,george s at the cove black bean soup,44123,90,35193,2002-10-25,NaN,an original recipe created by chef scott meska...,18.0
1,healthy for them yogurt popsicles,67664,10,91970,2003-07-26,NaN,my children and their friends ask for my homem...,NaN
2,i can t believe it s spinach,38798,30,1533,2002-08-29,NaN,"these were so go, it surprised even me.",8.0
3,italian gut busters,35173,45,22724,2002-07-27,NaN,my sister-in-law made these for us at a family...,NaN
4,love is in the air beef fondue sauces,84797,25,4470,2004-02-23,4.0,i think a fondue is a very romantic casual din...,NaN


## Измерение времени выполнения кода

Создайте версию таблицы, содержащие строки строки для рецептов, которые были добавлены в 2010 году.

Реализуйте несколько вариантов функции подсчета средней длины полного описания рецепта для рецептов, добавленных в 2010 году. Полным описанием рецепта называется строка, полученная путем конкатенации названия и описания рецепта через пробел.

In [ ]:
recipes.submitted = pd.to_datetime(recipes.submitted)
new_df = recipes[recipes.submitted.dt.year == 2010]
new_df.head()

,name,id,minutes,contributor_id,submitted,n_steps,description,n_ingredients
52,just peachy cobbler,437637,70,1085867,2010-09-17,10.0,all i can say is yummmmmm . . . a simple to ma...,10.0
68,the heat spicy party mix,437219,95,1682162,2010-09-13,NaN,a spicy chex mix that will really warm your gu...,11.0
81,iowa state fair sweet dough caramel cinnamon ...,435816,80,17803,2010-08-24,29.0,this was the winning entry at the 2010 iowa st...,NaN
104,1 minute blueberries cream,428566,2,1375473,2010-06-04,4.0,i was craving blueberry tonight but wanted non...,NaN
146,2 2 2 diet mocha,416599,5,789314,2010-03-15,5.0,"while trying to come up with a satisfying ""sna...",7.0


№1\.1 С использованием метода `DataFrame.iterrows` таблицы:

- функция принимает на вход таблицу, содержащую рецепты за 2010 год;

- вычисление полного описания рецепта осуществляется внутри цикла по `iterrows` для каждой строки по отдельности.

In [ ]:
def get_mean_len_A(df: pd.DataFrame) -> float:
    mask = []
    for _, row in df.iterrows():
        mask.append(len(row['name'] + ' ' + row['description']))
    return np.array(mask).mean()

In [ ]:
get_mean_len_A(new_df)

265.501300390117

№1\.2. С использованием метода `DataFrame.apply` таблицы:

- функция принимает на вход таблицу, содержащую рецепты за 2010 год;

- вызываете метод apply у таблицы; в качестве аргумента передаете функцию, которая возвращает длину полного описания для каждой строки;

- считаете среднюю длину описаний, вызвав соответствующий метод серии.

In [ ]:
def get_mean_len_B(df: pd.DataFrame) -> float:
    return df.description.apply(lambda x: len(x)).mean() + 1 + df.name.apply(len).mean()

In [ ]:
get_mean_len_B(new_df)

265.501300390117

№1\.3. С использованием векторизованных методов серий `pd.Series`:

- функция принимает на вход таблицу, содержащую рецепты за 2010 год;

- при помощи векторизованной операции сложения получаете столбец с полным описанием;

- считаете длину каждого элемента столбца с полным описанием, воспользовавшись соответствующим строковым методом аксессора `.str`;

- считаете среднюю длину описаний, вызвав соответствующий метод серии.

In [ ]:
def get_mean_len_C(df):
    df = df.assign(full_recipe = df.name + ' ' + df.description)
    return df.full_recipe.str.len().mean()

In [ ]:
get_mean_len_C(new_df)

265.501300390117

№1.4 Проверьте, что результаты работы всех написанных функций корректны и совпадают. Измерьте выполнения всех написанных функций при помощи магических команд `time` и `timeit`.

In [ ]:
%%timeit
get_mean_len_A(new_df)

59 ms ± 354 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
%%timeit
get_mean_len_B(new_df)

898 µs ± 5.47 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [ ]:
%%timeit
get_mean_len_C(new_df)

1.15 ms ± 15.9 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


## Анализ пошагового выполнения кода

Вам предлагается воспользоваться функцией, которая собирает статистику о том, сколько отзывов содержат то или иное слово.

In [ ]:
import re


def get_word_reviews_count(df):
    word_reviews = {}
    for review_id, row in df.dropna(subset=["review"]).iterrows():
        review = row["review"]
        words = re.sub(r"[^A-Za-z\s]", "", review).split(" ")
        for word in words:
            if word.lower() not in word_reviews:
                word_reviews[word.lower()] = set()
            word_reviews[word.lower()].add(review_id)
    word_reviews_count = {}
    for _, row in df.dropna(subset=["review"]).iterrows():
        review = row["review"]
        words = re.sub(r"[^A-Za-z\s]", "", review).split(" ")
        for word in words:
            word_reviews_count[word.lower()] = len(word_reviews[word.lower()])
    return word_reviews_count

№2.1 Найдите узкие места в коде, проанализировав код функции по шагам, используя профайлер. Сохраните результаты работы профайлера в отдельную текстовую ячейку. Выпишите (словами), что в имеющемся коде реализовано неоптимально.

In [ ]:
%%timeit
get_word_reviews_count(reviews)

15.4 s ± 62 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
%lprun -f get_word_reviews_count get_word_reviews_count(reviews)

Timer unit: 1e-09 s

Total time: 30.9721 s
File: <ipython-input-570-f37ec2aa0f5d>
Function: get_word_reviews_count at line 4

Line #      Hits         Time  Per Hit   % Time  Line Contents
     4                                           def get_word_reviews_count(df):
     5         1       2000.0   2000.0      0.0      word_reviews = {}
     6    126679 7736010000.0  61067.8     25.0      for review_id, row in df.dropna(subset=["review"]).iterrows():
     7    126679  670599000.0   5293.7      2.2          review = row["review"]
     8    126679 1169798000.0   9234.3      3.8          words = re.sub(r"[^A-Za-z\s]", "", review).split(" ")
     9   6792010 1060528000.0    156.1      3.4          for word in words:
    10   6712650 2171700000.0    323.5      7.0              if word.lower() not in word_reviews:
    11     79360   46128000.0    581.2      0.1                  word_reviews[word.lower()] = set()
    12   6792010 3640467000.0    536.0     11.8              word_reviews[word

### В данном коде реализованы неоптимально такие моменты как:
##### Код содержит множество вложенных циклов поверх цикла, который проходится по всему датасету, очевидно, что это не лушчий вариант программного кода. В данной функции есть два блока, первый для формирования словаря с уникальными айдишниками, второй для подсчета этих айдишников. Отсюда напрашивается второй блок убрать и из первого словаря по генератору перейти в нужный, минуя второй проход по таблице. Также, по ходу решения увидел, что при append код выполняется быстрее, чем через add, поэтому множества добавил только в генераторе. Существует ненужный словарь word_reviews_count. В первом варианте не удалось уйти от вложенных циклов и итерроус в целом, отсюда прирост только лишь в 2.5 раза.

№2.2  Оптимизируйте функцию и добейтесь значительного (как минимум, в 5 раз) прироста в скорости выполнения. Для демонстрации результата измерьте скорость выполнения оригинальной функции и функции, написанной вами.

## 1 способ

In [ ]:
import re

def get_word_reviews_count(df):
    word_reviews = {}
    df = df.dropna(subset=["review"])
    patt = re.compile(r"[^A-Za-z\s]")
    for review_id, row in df.iterrows():
        words = patt.sub("", row["review"]).lower().split(' ')
        for word in words:
            if word not in word_reviews:
                word_reviews[word] = []
            word_reviews[word].append(review_id)
    word_reviews = {key: len(set(value)) for key, value in word_reviews.items()}
    return word_reviews

In [ ]:
%%timeit
get_word_reviews_count(reviews)

6.69 s ± 93.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
%lprun -f get_word_reviews_count get_word_reviews_count(reviews)

Timer unit: 1e-09 s

Total time: 17.0409 s
File: <ipython-input-415-77d6fed6d9ae>
Function: get_word_reviews_count at line 4

Line #      Hits         Time  Per Hit   % Time  Line Contents
     4                                           def get_word_reviews_count(df):
     5         1       1000.0   1000.0      0.0      word_reviews = {}
     6         1   27757000.0 27757000.0      0.2      df = df.dropna(subset=["review"])
     7    126679 9666162000.0  76304.4     56.7      for review_id, row in df.iterrows():
     8    126679 1963799000.0  15502.2     11.5          words = re.sub(r"[^A-Za-z\s]", "", row["review"]).lower().split(" ")
     9   6792010 1081030000.0    159.2      6.3          for word in words:
    10   6712650 1756369000.0    261.7     10.3              if word not in word_reviews:
    11     79360   21478000.0    270.6      0.1                  word_reviews[word] = []
    12   6792010 2236291000.0    329.3     13.1              word_reviews[word].append(review_id)
 

## 2 способ из лекции по векторизации

In [ ]:
def get_word_reviews_count2(df):
    dct = {}
    patt = re.compile(r"[^A-Za-z\s]")
    for st in np.vectorize(lambda x: set(patt.sub("", x).lower().split(" ")))(reviews.review.dropna().to_numpy()):
        for word in st:
            if word not in dct:
                dct[word] = 0
            dct[word] += 1
    return dct

In [ ]:
%%timeit
get_word_reviews_count2(reviews)

2.27 s ± 63.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Counter method

In [ ]:
def word_counter(df):
    counter = Counter()
    patt = re.compile(r"[^A-Za-z\s]")
    df["review"].dropna().apply(lambda x: counter.update(set(patt.sub("", x).lower().split(" "))))
    return counter

In [ ]:
%%timeit
word_counter(reviews)

1.9 s ± 5.11 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Top 1 (крутой код, использующий только стек пандаса)

In [ ]:
%%timeit
reviews["review"].dropna().str.lower().replace(
    r"[^A-Za-z\s]", '', regex=True
).str.split(' ').apply(
    set).explode().value_counts(sort=False)

3.88 s ± 667 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
